In [74]:
# imports

import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import OpenAI, api_key
import requests
import pdfplumber
from io import BytesIO

# Restaurant Menu Analysis Tool

## Overview
This notebook demonstrates how to use **AI (OpenAI API)** to intelligently analyze restaurant menus. Specifically, it:
- Extracts text content from a PDF menu
- Uses GPT-4 to understand and analyze the menu
- Answers customer questions about menu items (e.g., "What's best for vegetarians?")

## Workflow
1. **Load API credentials** - Initialize OpenAI API connection
2. **Extract menu** - Download and extract text from PDF menu
3. **Process with AI** - Send menu to GPT-4 with a specific question
4. **Display results** - Show the AI's response in a readable format

This is useful for restaurants that want to help customers quickly find menu options matching their dietary preferences.

In [75]:
# Load environment variables from .env file
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    raise ValueError("API key not found. Please set the OPENAI_API_KEY environment variable.")
elif not api_key.startswith('sk-proj'):
    raise ValueError("Invalid API key format. Please ensure your API key starts with 'sk-proj'.")
elif api_key.strip() != api_key:
    raise ValueError("API key contains leading or trailing whitespace. Please remove any extra spaces.")
else:
    print("API key is valid and properly formatted.")

API key is valid and properly formatted.


In [76]:
#Define the system prompt for summarizing restaurant menus
system_prompt = """Eres un asistente de restaurante que ayuda a los clientes a entender el menu"""

In [77]:
#Define our user prompt

user_prompt_prefix = """Que postres tienes disponibles? \n\n
"""

In [78]:
# Define a function to extract text from a PDF menu given its URL. This function will download the PDF, extract the text using pdfplumber, and return the combined text from the specified number of pages.
def extract_menu_text_from_pdf(url):
    # Download the PDF file from the URL
    response = requests.get(url)
    pdf_file = BytesIO(response.content)

    # Extract text from the PDF using pdfplumber
    menu_text = ""
    with pdfplumber.open(pdf_file) as pdf:
        for page_num, page in enumerate(pdf.pages):  # Limit to the specified number of pages
            text = page.extract_text()
            menu_text += f"Page {page_num + 1}:\n{text}\n"
    return menu_text

In [79]:
# Define a function to create the messages for the OpenAI API call, combining the system prompt and the user prompt with the menu text.
def messages_for(menu_text):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + menu_text}
    ]

In [80]:
# Verify ollama is runnning and accessible at the expected endpoint. If this cell raises an error, please ensure that you have started the Ollama server by running `ollama serve` in your terminal.
requests.get("http://localhost:11434").content

b'Ollama is running'

In [81]:
# Define the main function to summarize the menu. This function will call the text extraction function, create the messages for the API call, and then use the OpenAI client to get the summary of the menu.
def summarize_menu(menu_url):
    # Extract the menu text from the PDF
    menu_text = extract_menu_text_from_pdf(menu_url)

    # Create the messages for the OpenAI API call
    messages = messages_for(menu_text)
    print(messages)

    OLLAMA_BASE_URL = "http://localhost:11434/v1"

    # Initialize the OpenAI client and make the API call to get the summary
    ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
    response = ollama.chat.completions.create(
        model="llama3.1:latest",
        messages=messages
    )

    # Extract and return the summary from the response
    summary = response.choices[0].message.content
    return summary

In [82]:
# Define a function to display the summary in a Markdown format. This function will call the summarize_menu function and then use IPython's display function to show the summary in a readable format.
def display_summary(url):
    summary = summarize_menu(url)
    display(Markdown(summary))


In [83]:
# Now we can call the display_summary function with the URL of the PDF menu to see the results.
url = "https://www.hotelesestelar.com/sites/all/themes/files/QR/HE025/04/01.pdf"
display_summary(url)

[{'role': 'system', 'content': 'Eres un asistente de restaurante que ayuda a los clientes a entender el menu'}, {'role': 'user', 'content': 'Que postres tienes disponibles? \n\n\nPage 1:\nM E N Ú\nRNT 19691\nPage 2:\nDESAYUNOS / BREAKFAST\nAMERICANO / AMERICAN $20.000\nJugo, fruta, huevos al gusto, pan, café o chocolate.\nJuice, fruit, eggs, bread, coffee or chocolate.\nBUFFET $28.000\nPARA PICAR / APPETIZERS\nAREPITAS RANCHERAS / CORN PATTIES $13.000\n2 unidades de arepita crocante de masa de maíz rellenas\nde salchicha y queso. 2 crispy corn patties filled with\nsausage and cheese.\nPATACONES GRATINADOS CON HOGAO $9.000\nFRIED GREEN PLANTAINS\nFried green plantains with cheese and a local tomato and\nonion sauce.\nMIX DE EMPANADAS DE RES Y POLLO $12.000\nFRIED PATTIES FILLED\n6 unidades de empanaditas tipo coctel con ají pico de\ngallo y cascos de limón. 6 fried patties filled with minced\nbeef or chicken.\nENTRADAS / APPETIZERS\nSOPA DEL HUERTO AROMATIZADA CON ALBAHACA $14.000\nVEGE

¡Claro! Nuestra propuesta de postres es muy variada y tenemos algo para todos los gustos. ¡Quieres que te muestre las opciones?

Tienes disponibles:

1. Copa de helado con salsa de frutos rojos ($9,000)
2. Torta de chocolate con helado ($9,000)
3. Torta de la casa con helado ($9,000)
4. Brevas con arequipe y queso ($9,000) ¡una auténtica delicia!

¿Cuál de estos postres te gustaría probar?